In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os

Mounted at /content/drive


In [2]:
!pip install transformers scikit-learn

In [3]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from pathlib import Path
import numpy as np

In [4]:
model_path = "/content/drive/MyDrive/rubert_tiny2"
tokenizer = AutoTokenizer.from_pretrained(model_path)
encoder = AutoModel.from_pretrained(model_path)

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

In [5]:
class RuBERTClassifier(nn.Module):
    def __init__(self, encoder, num_classes=7, dropout=0.3):
        super().__init__()
        self.encoder = encoder
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(312, num_classes)  # 312 — размер эмбеддингов rubert-tiny2

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        # Берём CLS-токен (как пулинг в GigaAM-Emo)
        cls_embedding = outputs.last_hidden_state[:, 0, :]
        x = self.dropout(cls_embedding)
        logits = self.classifier(x)
        return logits

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = RuBERTClassifier(encoder, num_classes=7)
model.to(device)

RuBERTClassifier(
  (encoder): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(83828, 312, padding_idx=0)
      (position_embeddings): Embedding(2048, 312)
      (token_type_embeddings): Embedding(2, 312)
      (LayerNorm): LayerNorm((312,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-2): 3 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=312, out_features=312, bias=True)
              (key): Linear(in_features=312, out_features=312, bias=True)
              (value): Linear(in_features=312, out_features=312, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=312, out_features=312, bias=True)
              (LayerNorm): LayerNorm((312,), eps=1e-12, elementwis

In [7]:
semantic_list = ['absent', 'command', 'illegible', 'insult-threat', 'other', 'question', 'statement']
semantic2id = {sem: i for i, sem in enumerate(semantic_list)}
id2semantic = {i: sem for sem, i in semantic2id.items()}

In [8]:
data_dir = Path("/content/drive/MyDrive/876/")
txt_path = data_dir / "ASR.txt"

texts = []
labels = []

with open(txt_path, 'r', encoding='utf-8') as f:
    for line in f:
        parts = line.strip().split(maxsplit=1)
        if len(parts) == 2:
            name, text = parts
            semantic = name.split('_')[0]
            if semantic in semantic2id:
                texts.append(text)
                labels.append(semantic2id[semantic])

In [9]:
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    texts, labels, test_size=0.4, random_state=42, stratify=labels
)
val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels, test_size=0.5, random_state=42, stratify=temp_labels
)

print(f"Train: {len(train_texts)}, Val: {len(val_texts)}, Test: {len(test_texts)}")

Train: 495, Val: 165, Test: 166


In [10]:
class SemanticDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(label, dtype=torch.long)
        }

In [11]:
train_dataset = SemanticDataset(train_texts, train_labels, tokenizer)
val_dataset = SemanticDataset(val_texts, val_labels, tokenizer)
test_dataset = SemanticDataset(test_texts, test_labels, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [12]:
model.load_state_dict(torch.load("/content/drive/MyDrive/rubert_semantic_best.pth"))

<All keys matched successfully>

In [13]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    'balanced',
    classes=np.unique(train_labels),
    y=train_labels
)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

best_val_loss = float('inf')
best_model_state = None

for epoch in range(5):
    # Train
    model.train()
    train_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # Validation
    model.eval()
    val_loss = 0
    val_preds, val_true = [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)
            val_loss += loss.item()
            preds = torch.argmax(logits, dim=1)
            val_preds.extend(preds.cpu().numpy())
            val_true.extend(labels.cpu().numpy())

    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    val_acc = accuracy_score(val_true, val_preds)
    val_f1 = f1_score(val_true, val_preds, average='weighted')

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_model_state = model.state_dict().copy()
        print(f"Лучшая модель (Val Loss: {best_val_loss:.4f})")

    print(f"Эпоха {epoch+1} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}")

Лучшая модель (Val Loss: 0.5170)
Эпоха 1 | Train Loss: 0.5930 | Val Loss: 0.5170 | Val Acc: 0.8121 | Val F1: 0.8094
Лучшая модель (Val Loss: 0.5147)
Эпоха 2 | Train Loss: 0.4644 | Val Loss: 0.5147 | Val Acc: 0.8061 | Val F1: 0.8030
Эпоха 3 | Train Loss: 0.4151 | Val Loss: 0.5154 | Val Acc: 0.8061 | Val F1: 0.8030
Лучшая модель (Val Loss: 0.5093)
Эпоха 4 | Train Loss: 0.4091 | Val Loss: 0.5093 | Val Acc: 0.8000 | Val F1: 0.7974
Эпоха 5 | Train Loss: 0.3683 | Val Loss: 0.5125 | Val Acc: 0.8121 | Val F1: 0.8095


In [15]:
torch.save(best_model_state, "/content/drive/MyDrive/rubert_semantic_best.pth")
print("Модель сохранена")

Модель сохранена


In [14]:
model.load_state_dict(best_model_state)
model.eval()
test_preds, test_true = [], []
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        logits = model(input_ids, attention_mask)
        preds = torch.argmax(logits, dim=1)
        test_preds.extend(preds.cpu().numpy())
        test_true.extend(labels.cpu().numpy())

test_acc = accuracy_score(test_true, test_preds)
test_f1 = f1_score(test_true, test_preds, average='weighted')
print(f"\n=== Тест ===")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test F1: {test_f1:.4f}")
print("\nClassification Report:")
print(classification_report(test_true, test_preds, target_names=semantic_list))


=== Тест ===
Test Accuracy: 0.8133
Test F1: 0.8129

Classification Report:
               precision    recall  f1-score   support

       absent       1.00      0.85      0.92        13
      command       0.79      0.85      0.81        26
    illegible       0.88      0.94      0.91        16
insult-threat       0.70      0.73      0.72        26
        other       0.88      0.84      0.86        25
     question       0.84      0.91      0.88        23
    statement       0.76      0.70      0.73        37

     accuracy                           0.81       166
    macro avg       0.84      0.83      0.83       166
 weighted avg       0.82      0.81      0.81       166



# Скачиваем на диск

In [ ]:
from transformers import AutoTokenizer, AutoModel

# Скачиваем модель и токенизатор на диск (в твою папку на Google Drive)
model_path = "/content/drive/MyDrive/rubert_tiny2"
tokenizer = AutoTokenizer.from_pretrained("cointegrated/rubert-tiny2")
model = AutoModel.from_pretrained("cointegrated/rubert-tiny2")

# Сохраняем на диск (чтобы в следующий раз не качать)
tokenizer.save_pretrained(model_path)
model.save_pretrained(model_path)

print(f"Модель сохранена в {model_path}")

config.json:   0%|          | 0.00/693 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/401 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/118M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Модель сохранена в /content/drive/MyDrive/rubert_tiny2
